In [0]:
datasets = {
    'daily_production': spark.read.parquet('/Volumes/workspace/default/raw_data/daily_production.parquet'),
    'production_failure': spark.read.parquet('/Volumes/workspace/default/raw_data/production_failures.parquet'),
    'well_jobs': spark.read.parquet('/Volumes/workspace/default/raw_data/well_jobs.parquet'),
}

for name, df in datasets.items():
  df.createOrReplaceTempView(name)

In [0]:
%sql
--DISPLAY FIRST 20 ROWS OF DATASET (daily_production)
SELECT * FROM daily_production LIMIT 10;

In [0]:
%sql
--DISPLAY FIRST 10 ROWS OF DATASET (production_failures)
SELECT * FROM production_failure LIMIT 20;
    

In [0]:
%sql
--DISPLAY FIRST 10 ROWS OF DATASET (well_jobs)
SELECT * FROM well_jobs LIMIT 10; 

In [0]:
%sql
-- pivot table showing the number of each job_category and total job done for each uwi
WITH cleaned AS (
    SELECT
        UPPER(TRIM(uwi)) AS uwi_clean,
        TRIM(job_category) AS job_category_clean
    FROM well_jobs
    WHERE uwi IS NOT NULL
      AND TRIM(uwi) <> ''
      AND UPPER(TRIM(uwi)) NOT IN ('NA', 'N/A', 'NULL', 'NONE')
)
SELECT
    uwi_clean AS uwi,
    COUNT(*) FILTER (WHERE LOWER(job_category_clean) = 'rod truck') AS rod_truck_count,
    COUNT(*) FILTER (WHERE LOWER(job_category_clean) = 'workover') AS workover_count,
    COUNT(*) FILTER (WHERE LOWER(job_category_clean) = 'drilling') AS drilling_count,
    COUNT(*) FILTER (WHERE LOWER(job_category_clean) = 'completion') AS completion_count,
    COUNT(*) FILTER (WHERE LOWER(job_category_clean) = 'abandon') AS abandon_count,
    COUNT(*) FILTER (
        WHERE job_category_clean IS NULL
           OR job_category_clean = ''
           OR LOWER(job_category_clean) NOT IN ('rod truck', 'workover', 'drilling', 'completion', 'abandon')
    ) AS other_or_unspecified_count,
    COUNT(*) AS total_jobs
FROM cleaned
GROUP BY uwi_clean
ORDER BY total_jobs DESC, uwi_clean;

In [0]:
%sql
WITH distinct_pump_uwi AS (
    SELECT DISTINCT
        UPPER(TRIM(uwi)) AS uwi_clean,
        TRIM(pcp_pump_make) AS pcp_pump_make,
        TRIM(pcp_pump_model) AS pcp_pump_model
    FROM daily_production
    WHERE uwi IS NOT NULL
      AND TRIM(uwi) <> ''
      AND pcp_pump_make IS NOT NULL
      AND TRIM(pcp_pump_make) <> ''
      AND pcp_pump_model IS NOT NULL
      AND TRIM(pcp_pump_model) <> ''
),
classified_jobs AS (
    SELECT
        UPPER(TRIM(uwi)) AS uwi_clean,
        CASE
            WHEN LOWER(job_type) LIKE '%polish rod%' OR LOWER(job_type) LIKE '%parted rod%'
            OR LOWER(job_type) LIKE '%broken rod%'
            OR LOWER(job_type) LIKE '%rod parted%'
            OR LOWER(job_type) LIKE '%rod break%'
            OR LOWER(job_type) LIKE '%sucker rod%'
                THEN 'Rods / Polish Rod'
            WHEN LOWER(job_type) LIKE '%tubing leak%'
            OR LOWER(job_type) LIKE '%tubing split%'
            OR LOWER(job_type) LIKE '%hole in tbg%' OR LOWER(job_type) LIKE '%tbg leak%'
            OR LOWER(job_type) LIKE '%tbg split%'
            OR LOWER(job_type) LIKE '%corroded tbg%'
                THEN 'Tubing Leak / Split'
            WHEN LOWER(job_type) LIKE '%stator%'
            OR LOWER(job_type) LIKE '%rotor%'
            OR LOWER(job_type) LIKE '%pump wear%'
            OR LOWER(job_type) LIKE '%worn pump%'
            OR LOWER(job_type) LIKE '%seized%'
            OR LOWER(job_type) LIKE '%stuck pump%'
            OR LOWER(job_type) LIKE '%pump change%'
                THEN 'Pump / Rotor / Stator'
            WHEN LOWER(job_type) LIKE '%packer%'
            OR LOWER(job_type) LIKE '%bridge plug%'
            OR LOWER(job_type) LIKE '%cbp%'
            OR LOWER(job_type) LIKE '%pbp%'
            OR LOWER(job_type) LIKE '%plug%'
                THEN 'Packer / Bridge Plug'
            WHEN LOWER(job_type) LIKE '%sand%'
            OR LOWER(job_type) LIKE '%sanded%'
            OR LOWER(job_type) LIKE '%sand fill%'
            OR LOWER(job_type) LIKE '%cleanout%'
            OR LOWER(job_type) LIKE '%sco%'
                THEN 'Sand / Fill / Cleanout'
            ELSE 'Other / Unclassified'
        END AS job_type
    FROM well_jobs
    WHERE uwi IS NOT NULL AND TRIM(uwi) <> ''
)
SELECT
    dp.pcp_pump_make,
    dp.pcp_pump_model,
    COUNT(*) FILTER (WHERE cj.job_type = 'Rods / Polish Rod') AS rods_polish_rod,
    COUNT(*) FILTER (WHERE cj.job_type = 'Tubing Leak / Split') AS tubing_leak_split,
    COUNT(*) FILTER (WHERE cj.job_type = 'Pump / Rotor / Stator') AS pump_rotor_stator,
    COUNT(*) FILTER (WHERE cj.job_type = 'Packer / Bridge Plug') AS packer_bridge_plug,
    COUNT(*) FILTER (WHERE cj.job_type = 'Sand / Fill / Cleanout') AS sand_fill_cleanout,
    COUNT(*) FILTER (WHERE cj.job_type = 'Other / Unclassified') AS other_unclassified,
    COUNT(*) AS total_jobs
FROM classified_jobs AS cj
LEFT JOIN distinct_pump_uwi AS dp
    ON cj.uwi_clean = dp.uwi_clean
GROUP BY
    dp.pcp_pump_make,
    dp.pcp_pump_model
ORDER BY
    total_jobs DESC,
    dp.pcp_pump_make,
    dp.pcp_pump_model;

In [0]:
%sql

WITH distinct_wells AS (
    SELECT DISTINCT
        uwi,
        pcp_pump_make AS make,
        pcp_pump_model AS model
    FROM daily_production
    WHERE uwi IS NOT NULL
      AND pcp_pump_make IS NOT NULL
      AND pcp_pump_model IS NOT NULL
),
well_job_est AS (
    SELECT
        dw.uwi,
        dw.make,
        dw.model,
        AVG(wj.field_est_total) AS avg_est_per_uwi,
        AVG(CASE WHEN wj.job_duration_days >= 0 THEN wj.job_duration_days END) AS avg_job_duration
    FROM well_jobs AS wj
    INNER JOIN distinct_wells AS dw
        ON wj.uwi = dw.uwi
    WHERE wj.field_est_total IS NOT NULL
    GROUP BY
        dw.uwi,
        dw.make,
        dw.model
)
SELECT
    make,
    model,
    ROUND(AVG(avg_est_per_uwi), 1) AS avg_estimate_cost_total,
    ROUND(AVG(avg_job_duration)) AS avg_job_duration
FROM well_job_est
GROUP BY
    make,
    model
ORDER BY
    avg_estimate_cost_total,
    avg_job_duration ASC,
    make,
    model;

In [0]:
%sql
-- 
WITH well_production AS (
    SELECT
        uwi,
        ROUND(SUM(oil_produced_m3), 3) AS total_oil_produced,
        ROUND(SUM(gas_produced_e3m3), 3) AS total_gas_produced,
        ROUND(SUM(lost_oil_m3), 3) AS total_lost_oil,
        ROUND(SUM(oil_produced_m3 - lost_oil_m3), 3) AS net_gain_m3
    FROM daily_production
    GROUP BY uwi
),

pump_info AS (
    SELECT DISTINCT
        uwi,
        pcp_pump_make,
        pcp_pump_model,
        pump_efficiency_pct
    FROM daily_production
)

SELECT
    p.uwi,
    w.pcp_pump_make,
    w.pcp_pump_model,
    p.total_oil_produced,
    p.total_gas_produced,
    p.total_lost_oil,
    p.net_gain_m3,
    ROUND(w.pump_efficiency_pct, 3) AS pump_efficientcy_pct
FROM well_production p
LEFT JOIN pump_info w
    ON p.uwi = w.uwi
ORDER BY p.net_gain_m3 DESC;

In [0]:
%sql
-- Top 5 pump models average efficiency
SELECT
    pcp_pump_make,
    pcp_pump_model,
    ROUND(AVG(pump_efficiency_pct), 3) AS avg_efficiency_pct
FROM daily_production
WHERE pump_efficiency_pct IS NOT NULL
GROUP BY
    pcp_pump_make,
    pcp_pump_model
ORDER BY avg_efficiency_pct DESC;

In [0]:
%sql
SELECT
    wj.uwi,
    dp.pcp_pump_make AS make,
    dp.pcp_pump_model AS model,
    wj.job_duration_days,
    wj.field_est_total
FROM well_jobs AS wj
LEFT JOIN (
    SELECT DISTINCT uwi, pcp_pump_make, pcp_pump_model
    FROM daily_production
    WHERE uwi IS NOT NULL
) AS dp
    ON wj.uwi = dp.uwi
WHERE wj.job_duration_days < 0
ORDER BY wj.job_duration_days ASC;

In [0]:
%sql
--number of pumps
SELECT
    province,
    pcp_pump_make,
    pcp_pump_model,
    COUNT(DISTINCT uwi) AS total_pumps
FROM daily_production
WHERE province IS NOT NULL
  AND pcp_pump_make IS NOT NULL
  AND pcp_pump_model IS NOT NULL
  AND uwi IS NOT NULL
GROUP BY
    province,
    pcp_pump_make,
    pcp_pump_model
ORDER BY
    province ASC,
    total_pumps DESC,
    pcp_pump_make ASC,
    pcp_pump_model ASC;

In [0]:
%sql
--Total pump in each province
SELECT
    province,
    COUNT(DISTINCT uwi) AS total_wells
FROM daily_production
WHERE province IS NOT NULL
  AND uwi IS NOT NULL
GROUP BY
    province
ORDER BY
    total_wells DESC,
    province ASC;

In [0]:
%sql
--Each province production
SELECT
    province,
    SUM(oil_produced_m3) AS total_oil_produced,
    SUM(gas_produced_e3m3) AS total_gas_produced,
    SUM(lost_oil_m3) AS total_lost_oil,
    SUM(total_oil_produced - total_lost_oil) AS net_gain_m3
FROM daily_production
WHERE province IS NOT NULL
GROUP BY province
ORDER BY total_oil_produced DESC
LIMIT 10;
    

In [0]:
%sql
--total failures per pump make and model 
WITH well_pumps AS (
    SELECT DISTINCT
        uwi,
        pcp_pump_make AS make,
        pcp_pump_model AS model
    FROM daily_production
    WHERE uwi IS NOT NULL
      AND pcp_pump_make IS NOT NULL
      AND pcp_pump_model IS NOT NULL
)
SELECT
    wp.make,
    wp.model,
    LOWER(TRIM(pf.failure_type)) AS fail_type,
    COUNT(*) AS total_failure_count
FROM production_failure AS pf
INNER JOIN well_pumps AS wp
    ON pf.uwi = wp.uwi
WHERE pf.failure_type IS NOT NULL
  AND TRIM(pf.failure_type) <> ''
GROUP BY
    wp.make,
    wp.model,
    LOWER(TRIM(pf.failure_type))
ORDER BY
    total_failure_count DESC;

In [0]:
%sql
WITH well_pumps AS (
    SELECT DISTINCT
        uwi,
        TRIM(pcp_pump_make) AS make,
        TRIM(pcp_pump_model) AS model
    FROM daily_production
    WHERE uwi IS NOT NULL
      AND pcp_pump_make IS NOT NULL
      AND pcp_pump_model IS NOT NULL
),
failure_counts AS (
    SELECT
        CASE
            WHEN LOWER(TRIM(pf.failure_type)) LIKE '%rod%' THEN 'rods'
            WHEN LOWER(TRIM(pf.failure_type)) LIKE '%tubing%'
              OR LOWER(TRIM(pf.failure_type)) LIKE '%tbg%' THEN 'tubing'
            WHEN LOWER(TRIM(pf.failure_type)) LIKE '%pump%'
              OR LOWER(TRIM(pf.failure_type)) LIKE '%pcp%' THEN 'pump'
            ELSE LOWER(TRIM(pf.failure_type))
        END AS fail_type,
        wp.make,
        wp.model,
        COUNT(*) AS total_failure_count
    FROM production_failure AS pf
    INNER JOIN well_pumps AS wp
        ON pf.uwi = wp.uwi
    WHERE pf.failure_type IS NOT NULL
      AND TRIM(pf.failure_type) <> ''
    GROUP BY
        CASE
            WHEN LOWER(TRIM(pf.failure_type)) LIKE '%rod%' THEN 'rods'
            WHEN LOWER(TRIM(pf.failure_type)) LIKE '%tubing%'
              OR LOWER(TRIM(pf.failure_type)) LIKE '%tbg%' THEN 'tubing'
            WHEN LOWER(TRIM(pf.failure_type)) LIKE '%pump%'
              OR LOWER(TRIM(pf.failure_type)) LIKE '%pcp%' THEN 'pump'
            ELSE LOWER(TRIM(pf.failure_type))
        END,
        wp.make,
        wp.model
)
SELECT
    fail_type,
    make,
    model,
    total_failure_count
FROM failure_counts
ORDER BY
    fail_type ASC,
    total_failure_count DESC,
    make ASC,
    model ASC;

In [0]:
%sql
-- total failure of each pump make and model
WITH well_pumps AS (
    SELECT DISTINCT
        uwi,
        TRIM(pcp_pump_make) AS make,
        TRIM(pcp_pump_model) AS model
    FROM daily_production
    WHERE uwi IS NOT NULL
      AND pcp_pump_make IS NOT NULL
      AND pcp_pump_model IS NOT NULL
)
SELECT
    wp.make,
    wp.model,
    COUNT(*) AS total_failure_count
FROM production_failure AS pf
INNER JOIN well_pumps AS wp
    ON pf.uwi = wp.uwi
WHERE pf.failure_type IS NOT NULL
  AND TRIM(pf.failure_type) <> ''
GROUP BY
    wp.make,
    wp.model
ORDER BY
    total_failure_count DESC,
    wp.make ASC,
    wp.model ASC;

In [0]:
%sql
WITH distinct_pump_uwi AS (
    SELECT DISTINCT
        UPPER(TRIM(uwi)) AS uwi_clean,
        TRIM(pcp_pump_make) AS make,
        TRIM(pcp_pump_model) AS model
    FROM daily_production
    WHERE uwi IS NOT NULL
      AND TRIM(uwi) <> ''
      AND pcp_pump_make IS NOT NULL
      AND TRIM(pcp_pump_make) <> ''
      AND pcp_pump_model IS NOT NULL
      AND TRIM(pcp_pump_model) <> ''
),
well_job_est AS (
    SELECT
        dp.uwi_clean,
        dp.make,
        dp.model,
        AVG(wj.field_est_total) AS avg_est_per_uwi,
        AVG(CASE WHEN wj.job_duration_days >= 0 THEN wj.job_duration_days END) AS avg_job_duration
    FROM well_jobs AS wj
    INNER JOIN distinct_pump_uwi AS dp
        ON UPPER(TRIM(wj.uwi)) = dp.uwi_clean
    WHERE wj.field_est_total IS NOT NULL
    GROUP BY
        dp.uwi_clean,
        dp.make,
        dp.model
),
maint_cost_duration AS (
    SELECT
        make,
        model,
        ROUND(AVG(avg_est_per_uwi), 1) AS avg_estimate_cost_total,
        ROUND(AVG(avg_job_duration)) AS avg_job_duration
    FROM well_job_est
    GROUP BY
        make,
        model
),
classified_jobs AS (
    SELECT
        UPPER(TRIM(uwi)) AS uwi_clean,
        CASE
            WHEN LOWER(job_type) LIKE '%polish rod%' OR LOWER(job_type) LIKE '%parted rod%'
            OR LOWER(job_type) LIKE '%broken rod%'
            OR LOWER(job_type) LIKE '%rod parted%'
            OR LOWER(job_type) LIKE '%rod break%'
            OR LOWER(job_type) LIKE '%sucker rod%'
                THEN 'Rods / Polish Rod'
            WHEN LOWER(job_type) LIKE '%tubing leak%'
            OR LOWER(job_type) LIKE '%tubing split%'
            OR LOWER(job_type) LIKE '%hole in tbg%' OR LOWER(job_type) LIKE '%tbg leak%'
            OR LOWER(job_type) LIKE '%tbg split%'
            OR LOWER(job_type) LIKE '%corroded tbg%'
                THEN 'Tubing Leak / Split'
            WHEN LOWER(job_type) LIKE '%stator%'
            OR LOWER(job_type) LIKE '%rotor%'
            OR LOWER(job_type) LIKE '%pump wear%'
            OR LOWER(job_type) LIKE '%worn pump%'
            OR LOWER(job_type) LIKE '%seized%'
            OR LOWER(job_type) LIKE '%stuck pump%'
            OR LOWER(job_type) LIKE '%pump change%'
                THEN 'Pump / Rotor / Stator'
            WHEN LOWER(job_type) LIKE '%packer%'
            OR LOWER(job_type) LIKE '%bridge plug%'
            OR LOWER(job_type) LIKE '%cbp%'
            OR LOWER(job_type) LIKE '%pbp%'
            OR LOWER(job_type) LIKE '%plug%'
                THEN 'Packer / Bridge Plug'
            WHEN LOWER(job_type) LIKE '%sand%'
            OR LOWER(job_type) LIKE '%sanded%'
            OR LOWER(job_type) LIKE '%sand fill%'
            OR LOWER(job_type) LIKE '%cleanout%'
            OR LOWER(job_type) LIKE '%sco%'
                THEN 'Sand / Fill / Cleanout'
            ELSE 'Other / Unclassified'
        END AS job_category
    FROM well_jobs
    WHERE uwi IS NOT NULL AND TRIM(uwi) <> ''
),
job_counts_by_model AS (
    SELECT
        dp.make,
        dp.model,
        COUNT(*) FILTER (WHERE cj.job_category = 'Rods / Polish Rod') AS rods_polish_rod,
        COUNT(*) FILTER (WHERE cj.job_category = 'Tubing Leak / Split') AS tubing_leak_split,
        COUNT(*) FILTER (WHERE cj.job_category = 'Pump / Rotor / Stator') AS pump_rotor_stator,
        COUNT(*) FILTER (WHERE cj.job_category = 'Packer / Bridge Plug') AS packer_bridge_plug,
        COUNT(*) FILTER (WHERE cj.job_category = 'Sand / Fill / Cleanout') AS sand_fill_cleanout,
        COUNT(*) FILTER (WHERE cj.job_category = 'Other / Unclassified') AS other_unclassified,
        COUNT(*) AS total_jobs
    FROM classified_jobs AS cj
    INNER JOIN distinct_pump_uwi AS dp
        ON cj.uwi_clean = dp.uwi_clean
    GROUP BY
        dp.make,
        dp.model
),
pump_efficiency AS (
    SELECT
        TRIM(pcp_pump_make) AS make,
        TRIM(pcp_pump_model) AS model,
        ROUND(AVG(pump_efficiency_pct), 3) AS avg_efficiency_pct
    FROM daily_production
    WHERE pump_efficiency_pct IS NOT NULL
      AND pcp_pump_make IS NOT NULL
      AND TRIM(pcp_pump_make) <> ''
      AND pcp_pump_model IS NOT NULL
      AND TRIM(pcp_pump_model) <> ''
    GROUP BY
        TRIM(pcp_pump_make),
        TRIM(pcp_pump_model)
)
SELECT
    COALESCE(jc.make, mcd.make, pe.make) AS make,
    COALESCE(jc.model, mcd.model, pe.model) AS model,
    pe.avg_efficiency_pct,
    mcd.avg_estimate_cost_total,
    mcd.avg_job_duration,
    COALESCE(jc.rods_polish_rod, 0) AS rods_polish_rod,
    COALESCE(jc.tubing_leak_split, 0) AS tubing_leak_split,
    COALESCE(jc.pump_rotor_stator, 0) AS pump_rotor_stator,
    COALESCE(jc.packer_bridge_plug, 0) AS packer_bridge_plug,
    COALESCE(jc.sand_fill_cleanout, 0) AS sand_fill_cleanout,
    COALESCE(jc.other_unclassified, 0) AS other_unclassified,
    COALESCE(jc.total_jobs, 0) AS total_jobs
FROM job_counts_by_model AS jc
FULL OUTER JOIN maint_cost_duration AS mcd
    ON jc.make = mcd.make
   AND jc.model = mcd.model
FULL OUTER JOIN pump_efficiency AS pe
    ON COALESCE(jc.make, mcd.make) = pe.make
   AND COALESCE(jc.model, mcd.model) = pe.model
WHERE COALESCE(jc.make, mcd.make, pe.make) IS NOT NULL
  AND TRIM(COALESCE(jc.make, mcd.make, pe.make)) <> ''
  AND COALESCE(jc.model, mcd.model, pe.model) IS NOT NULL
  AND TRIM(COALESCE(jc.model, mcd.model, pe.model)) <> ''
ORDER BY
    total_jobs DESC,
    avg_efficiency_pct DESC NULLS LAST,
    avg_estimate_cost_total ASC NULLS LAST,
    make ASC,
    model ASC;